### Package versions

Run the cell below to record the exact package versions used to generate the results in this notebook.


In [ ]:
# --- Environment / package versions (added for reproducibility) ---
import sys
try:
    from importlib.metadata import version, PackageNotFoundError
except ImportError:  # Python < 3.8
    from importlib_metadata import version, PackageNotFoundError
print("Python:", sys.version.split()[0])
for _pkg in ["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "seaborn", "gseapy", "pyliftover", "openpyxl", "requests", "pyBigWig", "pyreadr", "adjustText", "pypdf"]:
    try:
        print(f"{_pkg}: {version(_pkg)}")
    except PackageNotFoundError:
        pass


# Table 1 — GRCh37 → GRCh38 coordinate conversion

## Adjudication rule

A coordinate is accepted only when **at least two independent rsID-based sources agree**
(Ensembl and dbSNP), and the reference base at the resulting GRCh38 position matches the
expected reference allele. Chain liftOver is a third, positional opinion. Any disagreement
is reported per-variant and **blocks table generation**.

## Network requirement

Needs `rest.ensembl.org` and `eutils.ncbi.nlm.nih.gov`. The chain file
(`hgdownload.soe.ucsc.edu`) is optional — if it times out again, the notebook continues on
two sources and says so. Run locally; a network-restricted sandbox cannot execute this.


## 1. Setup

In [ ]:
# pip install requests pandas openpyxl        # pyliftover optional
import hashlib, json, sys, time, platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

OUTDIR   = Path(".")
ENSEMBL  = "https://rest.ensembl.org"
EUTILS   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
NCBI_API_KEY = None          # optional; raises the EUtils rate limit from 3/s to 10/s
CONTACT_EMAIL = "i.dos-santos-cebola@imperial.ac.uk"   # EUtils etiquette

CHAIN_URLS = [
    "https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz",
    "https://hgdownload-euro.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz",
]
CHAIN_LOCAL = OUTDIR / "hg19ToHg38.over.chain.gz"

PRIMARY = {str(c) for c in range(1, 23)} | {"X", "Y", "MT"}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": f"SLC38A4-manuscript-liftover ({CONTACT_EMAIL})"})

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)

# ------------------------------------------------------- dataframe schemas
# Each parser returns different keys depending on which branch it takes, so a
# column can silently fail to exist when no row happens to populate it. That is
# what broke section 7 previously: ref_at_dbsnp_pos is only set when two sources
# disagree, so when they all agreed the column was never created. Declaring the
# schema up front and reindexing makes every column always present.
ENS_COLS = ["rsid","ens_current_rsid","ens_merged","ens_chr","ens_pos_grch38",
            "ens_allele_string","ens_ref_allele","ens_n_alleles","ens_multi_allelic",
            "ens_strand","ens_all_positions","ens_status"]
DBSNP_COLS = ["rsid","dbsnp_current_rsid","dbsnp_merged","dbsnp_chr","dbsnp_pos_grch38",
              "dbsnp_chr37","dbsnp_pos_grch37","dbsnp_spdi","dbsnp_status"]
LO_COLS  = ["rsid","lo_chr","lo_pos_grch38","lo_strand","lo_status"]
REF_COLS = ["rsid","ref_at_ensembl_pos","ref_at_dbsnp_pos","ens_ref_allele",
            "ensembl_pos_ref_matches"]

BOOL_COLS = ["ens_merged","ens_multi_allelic","dbsnp_merged","ensembl_pos_ref_matches"]

def frame(records, columns, name=""):
    """Build a DataFrame with a guaranteed column set and stable dtypes."""
    df = pd.DataFrame(list(records)).reindex(columns=columns)
    for c in [c for c in BOOL_COLS if c in df.columns]:
        df[c] = df[c].fillna(False).astype(bool)
    for c in [c for c in df.columns if c.endswith(("_grch38","_grch37","_alleles"))]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    missing = [c for c in columns if df[c].isna().all()]
    if missing and name:
        print(f"  note: {name} has no values for {missing}")
    return df


# ---------------------------------------------------------------- HTTP helper
# Every API call goes through this. It checks status and Content-Type BEFORE
# parsing, and on failure shows what actually came back - which is how you tell
# a rate-limit from an institutional proxy page from a real outage.
class ApiError(RuntimeError):
    pass

def _describe(r):
    body = (r.text or "")[:300].replace("\n", " ")
    return (f"HTTP {r.status_code}, Content-Type={r.headers.get('Content-Type','?')}, "
            f"body[:300]={body!r}")

def request_json(method, url, *, retries=4, timeout=60, **kw):
    last = None
    for attempt in range(retries):
        try:
            r = SESSION.request(method, url, timeout=timeout, **kw)
        except requests.RequestException as e:
            last = f"{type(e).__name__}: {str(e)[:150]}"
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", 2 ** attempt))
            print(f"    rate limited, sleeping {wait}s"); time.sleep(wait)
            last = _describe(r); continue
        if r.status_code in (500, 502, 503, 504):
            last = _describe(r); time.sleep(2 ** attempt); continue
        if r.status_code != 200:
            raise ApiError(f"{method} {url}\n    {_describe(r)}")
        if "json" not in (r.headers.get("Content-Type") or "").lower():
            raise ApiError(
                f"{method} {url} returned non-JSON\n    {_describe(r)}\n"
                "    If the body looks like HTML, a proxy or captive portal is "
                "intercepting the request - try off the institutional network / VPN.")
        try:
            return r.json()
        except ValueError as e:
            raise ApiError(f"{method} {url} JSON parse failed: {e}\n    {_describe(r)}")
    raise ApiError(f"{method} {url} failed after {retries} attempts. Last: {last}")

print("python  ", sys.version.split()[0])
print("pandas  ", pd.__version__)
print("platform", platform.platform())


### 1b. Preflight connectivity check

Run this first. It reports exactly what each endpoint returns, so a failure tells you
*which* problem you have rather than surfacing as a `JSONDecodeError` twenty cells later.

Only the first two rows must pass. `hgdownload` is optional.


In [ ]:
ENDPOINTS = [
    ("Ensembl  ping",        f"{ENSEMBL}/info/ping",          {"content-type":"application/json"}, True),
    ("dbSNP    esummary",    f"{EUTILS}/esummary.fcgi",
         {"db":"snp","id":"738409","retmode":"json"},                                              True),
    ("Ensembl  release",     f"{ENSEMBL}/info/data",          {"content-type":"application/json"}, False),
    ("Ensembl  sequence",    f"{ENSEMBL}/sequence/region/human/22:43928847..43928847",
         {"coord_system_version":"GRCh38","content-type":"application/json"},                      False),
    ("UCSC     chain (opt)", CHAIN_URLS[0],                   None,                                False),
]

required_ok = True
for label, url, params, required in ENDPOINTS:
    try:
        r = SESSION.get(url, params=params, timeout=(10, 30),
                        headers={"Accept":"application/json"} if params else None,
                        stream=(params is None))
        ct = (r.headers.get("Content-Type") or "?").split(";")[0]
        verdict = "OK" if r.status_code == 200 else "FAIL"
        extra = ""
        if r.status_code == 200 and params is not None and "json" not in ct.lower():
            verdict, extra = "FAIL", "  <- not JSON; likely a proxy/portal page"
        if r.status_code != 200:
            extra = "  " + (r.text or "")[:120].replace("\n", " ")
        print(f"{verdict:4s} {label:22s} HTTP {r.status_code}  {ct}{extra}")
        if required and verdict != "OK":
            required_ok = False
        r.close()
    except Exception as e:
        print(f"FAIL {label:22s} {type(e).__name__}: {str(e)[:100]}")
        if required:
            required_ok = False

print()
if required_ok:
    print("Both required sources reachable - proceed.")
else:
    print("A REQUIRED source is unreachable. Do not interpret partial results.\n"
          "  - non-JSON / HTML body   -> institutional proxy; try off-VPN or a home network\n"
          "  - HTTP 429               -> rate limited; wait a few minutes\n"
          "  - HTTP 5xx               -> service outage; retry later\n"
          "  - connect timeout        -> firewall blocking outbound HTTPS")


## 2. Input — Table 1 as currently published (GRCh37/hg19)

Rows 9–11 (`rs6028984`, `rs3959632`, `rs738409`) are transposed in the published table —
*Position* and *Nearest gene* are swapped relative to the other eight. They are entered
here in the correct logical order.


In [ ]:
table1_grch37 = pd.DataFrame(
    [
        ("rs4241122",   "2", 113678856, "IL37",      "G", 0.25, 0.36, 0.59, 0.46, 0.76, 4.1e-5),
        ("rs6444127",   "3", 186143744, "LINC02052", "T", 0.39, 0.49, 0.60, 0.47, 0.76, 2.6e-5),
        ("rs9472138",   "6",  43811762, "VEGFA",     "T", 0.35, 0.25, 1.72, 1.34, 2.22, 2.8e-5),
        ("rs505347",    "6", 103600715, "GRIK2",     "A", 0.48, 0.37, 1.66, 1.30, 2.12, 4.6e-5),
        ("rs11183620", "12",  47212370, "SLC38A4",   "A", 0.55, 0.43, 1.66, 1.30, 2.12, 4.2e-5),
        ("rs2246129",  "13",  44826634, "SMIM2",     "T", 0.46, 0.57, 0.60, 0.48, 0.76, 2.7e-5),
        ("rs985975",   "16",  60812501, "CDH8",      "C", 0.09, 0.16, 0.46, 0.32, 0.66, 3.1e-5),
        ("rs11573",    "19",  51359497, "KLK3",      "C", 0.37, 0.48, 0.60, 0.48, 0.76, 2.4e-5),
        ("rs6028984",  "20",  38883710, "LINC01370", "T", 0.25, 0.36, 0.57, 0.44, 0.73, 1.3e-5),
        ("rs3959632",  "22",  36668884, "APOL1",     "C", 0.25, 0.16, 1.89, 1.40, 2.54, 3.0e-5),
        ("rs738409",   "22",  44324727, "PNPLA3",    "G", 0.31, 0.18, 2.09, 1.58, 2.77, 2.7e-7),
    ],
    columns=["rsid","chr","pos_grch37","nearest_gene","effect_allele",
             "eaf_cases","eaf_controls","OR","ci_low","ci_high","p"],
)
assert table1_grch37["rsid"].is_unique
print(f"{len(table1_grch37)} variants loaded")
table1_grch37


## 3. Source A — Ensembl REST (GRCh38 native)

In [ ]:
def ensembl_variants(rsids, chunk=200):
    out = {}
    for i in range(0, len(rsids), chunk):
        batch = rsids[i:i+chunk]
        out.update(request_json(
            "POST", f"{ENSEMBL}/variation/homo_sapiens",
            headers={"Content-Type":"application/json", "Accept":"application/json"},
            json={"ids":batch, "pops":0, "genotypes":0, "phenotypes":0},
            timeout=120))
        time.sleep(0.2)
    return out


# --- provenance metadata: NON-FATAL. These two values only get written to the
# --- provenance file; they must never be able to abort the analysis.
ENS_RELEASE, ENS_ASSEMBLY = "unknown", "unknown"
try:
    info = request_json("GET", f"{ENSEMBL}/info/data",
                        params={"content-type":"application/json"},
                        headers={"Accept":"application/json"}, timeout=30)
    ENS_RELEASE = (info.get("releases") or ["unknown"])[0]
except ApiError as e:
    print("WARNING: could not read Ensembl release (provenance only, continuing)")
    print("  ", str(e).replace("\n", "\n   "))

try:
    asm = request_json("GET", f"{ENSEMBL}/info/assembly/homo_sapiens",
                       params={"content-type":"application/json"},
                       headers={"Accept":"application/json"}, timeout=30)
    ENS_ASSEMBLY = asm.get("assembly_name", "unknown")
except ApiError as e:
    print("WARNING: could not read Ensembl assembly (provenance only, continuing)")
    print("  ", str(e).replace("\n", "\n   "))

print(f"\nEnsembl release {ENS_RELEASE}, assembly {ENS_ASSEMBLY}")
if str(ENS_ASSEMBLY).startswith("GRCh38"):
    pass
elif ENS_ASSEMBLY == "unknown":
    print("  assembly unconfirmed from the info endpoint - not fatal: the reference-base\n"
          "  check in section 6 queries with coord_system_version=GRCh38 explicitly, so\n"
          "  the build is verified there regardless.")
else:
    raise RuntimeError(f"Ensembl is serving {ENS_ASSEMBLY}, not GRCh38 - stop and check")

# --- the actual data fetch ---
records = ensembl_variants(table1_grch37["rsid"].tolist())
print(f"{len(records)}/{len(table1_grch37)} rsIDs returned")


In [ ]:
PALINDROMES = [{"A","T"}, {"C","G"}]

def parse_ensembl(rsid, rec):
    if rec is None:
        return dict(rsid=rsid, ens_status="not_found")
    current  = rec.get("name", rsid)
    maps = [m for m in rec.get("mappings", [])
            if str(m.get("assembly_name","")).startswith("GRCh38")
            and str(m.get("seq_region_name")) in PRIMARY]
    if not maps:
        return dict(rsid=rsid, ens_current_rsid=current,
                    ens_merged=(current!=rsid), ens_status="no_primary_GRCh38_mapping")

    # report ALL primary mappings, not just the first
    positions = sorted({(str(m["seq_region_name"]), int(m["start"])) for m in maps})
    status = "ok" if len(positions)==1 else f"multi_mapping_n={len(positions)}"
    m = maps[0]
    alleles = [a for a in str(m.get("allele_string","")).split("/") if a]
    return dict(
        rsid=rsid,
        ens_current_rsid=current,
        ens_merged=(current!=rsid),
        ens_chr=str(m["seq_region_name"]),
        ens_pos_grch38=int(m["start"]),
        ens_allele_string=m.get("allele_string"),
        ens_ref_allele=alleles[0] if alleles else None,   # Ensembl lists REF first
        ens_n_alleles=len(alleles),
        ens_multi_allelic=len(alleles) > 2,
        ens_strand=m.get("strand"),
        ens_all_positions=";".join(f"{c}:{p}" for c,p in positions),
        ens_status=status,
    )

ens_df = frame((parse_ensembl(r, records.get(r)) for r in table1_grch37["rsid"]),
               ENS_COLS, "ens_df")
print(f"Ensembl usable positions: "
      f"{int(ens_df['ens_pos_grch38'].notna().sum())}/{len(ens_df)}")
print(f"multi-allelic records: {int(ens_df['ens_multi_allelic'].sum())}/{len(ens_df)}"
      "  <- why a membership-based allele check is uninformative here")
ens_df[["rsid","ens_chr","ens_pos_grch38","ens_allele_string","ens_ref_allele",
        "ens_n_alleles","ens_multi_allelic","ens_strand","ens_status"]]


## 4. Source B — NCBI dbSNP via EUtils `esummary`

This is the resource you checked by hand, so it is the right independent arbiter.

`esummary db=snp` returns two fields that between them settle the question:

- `chrpos` — position on the **current** assembly (GRCh38)
- `chrpos_prev_assm` — position on the **previous** assembly (GRCh37)

The second is valuable beyond adjudication: it lets us verify that Table 1's existing hg19
coordinates were right in the first place. v1 never checked its own input.


In [ ]:
def dbsnp_esummary(rsids, chunk=100, max_retries=4):
    """Fetch dbSNP summaries. Returns {rsid: {...}}."""
    out = {}
    for i in range(0, len(rsids), chunk):
        batch = [r.replace("rs","") for r in rsids[i:i+chunk]]
        params = {"db":"snp","id":",".join(batch),"retmode":"json","email":CONTACT_EMAIL}
        if NCBI_API_KEY: params["api_key"] = NCBI_API_KEY
        res = request_json("GET", f"{EUTILS}/esummary.fcgi",
                           params=params, timeout=90, retries=max_retries).get("result", {})
        for uid in res.get("uids", []):
            out[f"rs{uid}"] = res[uid]
        time.sleep(0.4 if not NCBI_API_KEY else 0.12)
    return out

def parse_dbsnp(rsid, rec):
    if rec is None or "error" in (rec or {}):
        return dict(rsid=rsid, dbsnp_status="not_found")
    def split_pos(v):
        if not v or ":" not in str(v): return (None, None)
        c, p = str(v).split(":")[:2]
        try:    return (c, int(p))
        except ValueError: return (c, None)
    c38, p38 = split_pos(rec.get("chrpos"))
    c37, p37 = split_pos(rec.get("chrpos_prev_assm"))
    current  = f"rs{rec.get('snp_id')}" if rec.get("snp_id") else rsid
    return dict(
        rsid=rsid,
        dbsnp_current_rsid=current,
        dbsnp_merged=(current != rsid),
        dbsnp_chr=c38, dbsnp_pos_grch38=p38,
        dbsnp_chr37=c37, dbsnp_pos_grch37=p37,
        dbsnp_spdi=rec.get("spdi"),
        dbsnp_status="ok" if p38 else "no_grch38_position",
    )

dbsnp_records = dbsnp_esummary(table1_grch37["rsid"].tolist())
print(f"{len(dbsnp_records)}/{len(table1_grch37)} returned by dbSNP")
dbsnp_df = frame((parse_dbsnp(r, dbsnp_records.get(r)) for r in table1_grch37["rsid"]),
                 DBSNP_COLS, "dbsnp_df")

n_ok = int(dbsnp_df["dbsnp_pos_grch38"].notna().sum())
print(f"dbSNP usable GRCh38 positions: {n_ok}/{len(dbsnp_df)}")
if n_ok == 0:
    print("\n*** WARNING: dbSNP returned no usable positions. Adjudication would fall\n"
          "    back to a SINGLE source (Ensembl), which is exactly what this notebook\n"
          "    exists to avoid. Fix this before trusting any coordinate. ***")
elif n_ok < len(dbsnp_df):
    print("    (variants without a dbSNP position cannot be accepted)")
dbsnp_df


## 5. Source C — chain-file liftOver

In [ ]:
def sha256(path, buf=1<<20):
    h = hashlib.sha256()
    with open(path,"rb") as fh:
        for b in iter(lambda: fh.read(buf), b""): h.update(b)
    return h.hexdigest()

CHAIN_OK, CHAIN_SHA256, CHAIN_NOTE = False, None, ""

if not CHAIN_LOCAL.exists():
    for url in CHAIN_URLS:
        try:
            print(f"trying {url}")
            resp = SESSION.get(url, timeout=(10, 300))     # (connect, read)
            resp.raise_for_status()
            CHAIN_LOCAL.write_bytes(resp.content)
            print("  downloaded")
            break
        except Exception as e:
            print(f"  failed: {type(e).__name__}: {str(e)[:90]}")

if CHAIN_LOCAL.exists() and CHAIN_LOCAL.stat().st_size > 1000:
    try:
        from pyliftover import LiftOver
        lo = LiftOver(str(CHAIN_LOCAL))
        CHAIN_SHA256, CHAIN_OK = sha256(CHAIN_LOCAL), True
        print(f"chain ok, sha256 {CHAIN_SHA256[:16]}...")
    except ImportError:
        CHAIN_NOTE = "pyliftover not installed"
    except Exception as e:
        CHAIN_NOTE = f"chain load failed: {e}"
else:
    CHAIN_NOTE = ("chain file unavailable (all mirrors failed). "
                  "Optionally download manually into this folder:\n  "
                  + CHAIN_URLS[0])

if not CHAIN_OK:
    print("\nLIFTOVER CROSS-CHECK SKIPPED -", CHAIN_NOTE)
    print("Proceeding on two independent rsID sources (Ensembl + dbSNP), which is "
          "sufficient for adjudication. This is recorded in the provenance file.")


In [ ]:
def liftover_one(chrom, pos_1based):
    """1-based in, 1-based out. Chain files are 0-based half-open."""
    res = lo.convert_coordinate(f"chr{chrom}", int(pos_1based) - 1)
    if res is None:  return dict(lo_status="chrom_not_in_chain")
    res = [r for r in res if r[0].replace("chr","") in PRIMARY]
    if not res:      return dict(lo_status="unmapped")
    if len(res) > 1: return dict(lo_status=f"ambiguous_n={len(res)}")
    return dict(lo_chr=res[0][0].replace("chr",""),
                lo_pos_grch38=int(res[0][1]) + 1,
                lo_strand=res[0][2], lo_status="ok")

if CHAIN_OK:
    lo_df = frame(({"rsid":r.rsid, **liftover_one(r.chr, r.pos_grch37)}
                   for r in table1_grch37.itertuples()), LO_COLS, "lo_df")
else:
    lo_df = frame(({"rsid":r, "lo_status":"skipped"} for r in table1_grch37["rsid"]),
                  LO_COLS, "lo_df")
lo_df


## 6. Reference-base verification

For each accepted GRCh38 position, fetch the actual reference base from the Ensembl sequence endpoint and confirm it equals the reference allele in the variant record. If a coordinate is wrong by even one base this will usually fail, because the neighbouring base rarely matches.


In [ ]:
def ref_base(chrom, pos, assembly="GRCh38"):
    """Reference base at a 1-based GRCh38 position. Returns None if unavailable."""
    try:
        j = request_json("GET", f"{ENSEMBL}/sequence/region/human/{chrom}:{pos}..{pos}",
                         params={"coord_system_version": assembly,
                                 "content-type": "application/json"},
                         headers={"Accept": "application/json"}, timeout=60)
    except ApiError as e:
        print(f"  ref_base {chrom}:{pos} unavailable - {str(e).splitlines()[0]}")
        return None
    return (j.get("seq") or "").upper() or None

dbs_idx = dbsnp_df.set_index("rsid")

checks = []
for r in ens_df.itertuples():
    pos_e = r.ens_pos_grch38
    # every key always present - never let a column vanish
    row = {"rsid": r.rsid, "ref_at_ensembl_pos": None, "ref_at_dbsnp_pos": None}

    if pd.notna(pos_e) and pd.notna(r.ens_chr):
        row["ref_at_ensembl_pos"] = ref_base(str(r.ens_chr), int(pos_e))

    # probe the dbSNP position too, whenever it exists and differs
    pos_d = dbs_idx.at[r.rsid, "dbsnp_pos_grch38"] if r.rsid in dbs_idx.index else pd.NA
    if pd.notna(pos_d) and (pd.isna(pos_e) or int(pos_d) != int(pos_e)):
        chr_d = dbs_idx.at[r.rsid, "dbsnp_chr"]
        if pd.notna(chr_d):
            row["ref_at_dbsnp_pos"] = ref_base(str(chr_d), int(pos_d))

    checks.append(row)
    time.sleep(0.15)

ref_df = (frame(checks, ["rsid","ref_at_ensembl_pos","ref_at_dbsnp_pos"], "ref_df")
          .merge(ens_df[["rsid","ens_ref_allele"]], on="rsid", how="left", validate="1:1"))

ref_df["ensembl_pos_ref_matches"] = (
    ref_df["ref_at_ensembl_pos"].astype("string").str.upper()
    == ref_df["ens_ref_allele"].astype("string").str.upper()
).fillna(False)

ref_df = ref_df.reindex(columns=REF_COLS)

n_probed = int(ref_df["ref_at_dbsnp_pos"].notna().sum())
print(f"\nreference bases fetched at {int(ref_df['ref_at_ensembl_pos'].notna().sum())}"
      f"/{len(ref_df)} Ensembl positions; "
      f"{n_probed} alternative dbSNP position(s) also probed"
      + (" (none needed - sources agreed)" if n_probed == 0 else ""))
ref_df


## 7. Adjudication

Acceptance requires **all** of:

1. Ensembl and dbSNP return the same chromosome and position
2. Neither reports the rsID as merged
3. Neither reports multi-mapping
4. The reference base at that position matches the record's reference allele
5. dbSNP's previous-assembly position matches the hg19 coordinate in Table 1 *(validates
   the input; a mismatch here means Table 1's hg19 value was already wrong)*

Chain liftOver, when available, is reported as a third opinion but is not required —
positional liftover is the weakest of the three and should not overrule two concordant
rsID-based sources.


In [ ]:
# schema guard - fail loudly and early if an upstream cell was skipped
for nm, df, cols in [("ens_df", ens_df, ENS_COLS), ("dbsnp_df", dbsnp_df, DBSNP_COLS),
                     ("lo_df", lo_df, LO_COLS),   ("ref_df", ref_df, REF_COLS)]:
    absent = [c for c in cols if c not in df.columns]
    assert not absent, f"{nm} is missing {absent} - re-run the cell that builds it"

m = (table1_grch37
     .merge(ens_df,   on="rsid", how="left", validate="1:1")
     .merge(dbsnp_df, on="rsid", how="left", validate="1:1")
     .merge(lo_df,    on="rsid", how="left", validate="1:1")
     .merge(ref_df.drop(columns=["ens_ref_allele"]),
            on="rsid", how="left", validate="1:1"))

m["sources_agree_38"] = (
    (m["ens_status"]=="ok") & (m["dbsnp_status"]=="ok")
    & (m["ens_chr"].astype(str) == m["dbsnp_chr"].astype(str))
    & (m["ens_pos_grch38"] == m["dbsnp_pos_grch38"])
)
m["input_hg19_confirmed"] = m["pos_grch37"] == m["dbsnp_pos_grch37"]
m["liftover_agrees"] = m["lo_pos_grch38"] == m["ens_pos_grch38"]

m["consensus_pos_grch38"] = m["ens_pos_grch38"].where(m["sources_agree_38"])
m["delta_ens_dbsnp"] = m["dbsnp_pos_grch38"] - m["ens_pos_grch38"]
m["offset_37_to_38"]  = m["consensus_pos_grch38"] - m["pos_grch37"]

as_bool = lambda s: s.fillna(False).astype(bool)

m["accept"] = (
    m["sources_agree_38"]
    & ~as_bool(m["ens_merged"]) & ~as_bool(m["dbsnp_merged"])
    & as_bool(m["ensembl_pos_ref_matches"])
    & m["input_hg19_confirmed"]
)

cols = ["rsid","nearest_gene","chr","pos_grch37","dbsnp_pos_grch37","input_hg19_confirmed",
        "ens_pos_grch38","dbsnp_pos_grch38","delta_ens_dbsnp","lo_pos_grch38",
        "ref_at_ensembl_pos","ref_at_dbsnp_pos","ens_ref_allele",
        "ensembl_pos_ref_matches","sources_agree_38","accept"]
m[cols]


In [ ]:
bad = m.loc[~m["accept"]]
if len(bad):
    print(f"*** {len(bad)} of {len(m)} variant(s) NOT accepted ***\n")
    for r in bad.itertuples():
        why = []
        if r.ens_status   != "ok": why.append(f"Ensembl: {r.ens_status}")
        if r.dbsnp_status != "ok": why.append(f"dbSNP: {r.dbsnp_status}")
        if not r.sources_agree_38:
            fmt = lambda v: f"{int(v):,}" if pd.notna(v) else "n/a"
            why.append(f"Ensembl {fmt(r.ens_pos_grch38)} vs dbSNP {fmt(r.dbsnp_pos_grch38)}"
                       + (f" (delta {int(r.delta_ens_dbsnp):+,})"
                          if pd.notna(r.delta_ens_dbsnp) else ""))
        if bool(r.ens_merged) or bool(r.dbsnp_merged):
            why.append("rsID merged")
        if not bool(r.ensembl_pos_ref_matches):
            why.append(f"ref base {r.ref_at_ensembl_pos!r} != expected {r.ens_ref_allele!r}")
        if not r.input_hg19_confirmed:
            why.append(f"Table 1 hg19 {int(r.pos_grch37):,} != dbSNP prev-assembly "
                       + (f"{int(r.dbsnp_pos_grch37):,}"
                          if pd.notna(r.dbsnp_pos_grch37) else "n/a"))
        print(f"  {r.rsid:12s} ({r.nearest_gene:9s})  " + "; ".join(why))
    print("\nDo NOT put these rows into Table 1 until resolved.")
else:
    print(f"All {len(m)} variants accepted by two independent sources "
          "with reference-base confirmation.")

if CHAIN_OK:
    dis = m.loc[m["accept"] & ~m["liftover_agrees"].fillna(True), "rsid"].tolist()
    print(("\nliftOver disagrees with the consensus for: " + ", ".join(dis)) if dis
          else "\nliftOver agrees with the consensus everywhere.")
else:
    print(f"\nliftOver skipped ({CHAIN_NOTE.splitlines()[0]}).")

ma = m.loc[m["ens_multi_allelic"].fillna(False), "rsid"].tolist()
if ma:
    print("\nMulti-allelic in Ensembl (allele-membership checks are uninformative; "
          "reference-base check used instead): " + ", ".join(ma))

print("\nOffsets GRCh37 -> GRCh38:")
print(m[["rsid","chr","offset_37_to_38"]].to_string(index=False))


## 8. Output


In [ ]:
final = pd.DataFrame({
    "SNP": m["rsid"],
    "Chr": m["ens_chr"].fillna(m["chr"]),
    "Position (GRCh38/hg38)": m["consensus_pos_grch38"],
    "Position (GRCh37/hg19)": m["pos_grch37"],
    "Nearest gene": m["nearest_gene"],
    "EA": m["effect_allele"],
    "EAF cases": m["eaf_cases"], "EAF controls": m["eaf_controls"],
    "OR": m["OR"],
    "95% CI": m["ci_low"].map("{:.2f}".format) + "–" + m["ci_high"].map("{:.2f}".format),
    "p": m["p"],
    "QC": m["accept"].map({True:"pass", False:"REVIEW"}),
})
disp = final.copy()
for c in ["Position (GRCh38/hg38)","Position (GRCh37/hg19)"]:
    disp[c] = disp[c].map(lambda v: f"{int(v):,}" if pd.notna(v) else "NA")

final.to_csv(OUTDIR/"Table1_GRCh38_coordinates.csv", index=False)
try: disp.to_excel(OUTDIR/"Table1_GRCh38_coordinates.xlsx", index=False)
except ImportError: print("openpyxl missing - CSV only")
m.to_csv(OUTDIR/"Table1_liftover_full_audit.csv", index=False)

def _pyliftover_version():
    try:
        from importlib.metadata import version
        return version("pyliftover")
    except Exception:
        return "not installed"

provenance = {
  "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
  "notebook_version": "v2",
  "supersedes": "_run_log/Table1_liftover_run1_incomplete.ipynb (aborted at chain download)",
  "source_assembly":"GRCh37/hg19", "target_assembly":"GRCh38/hg38",
  "sources": {
    "ensembl": {"endpoint":ENSEMBL, "release":ENS_RELEASE, "assembly":ENS_ASSEMBLY},
    "dbsnp":   {"endpoint":f"{EUTILS}/esummary.fcgi", "db":"snp"},
    "chain_liftover": ({"urls":CHAIN_URLS, "sha256":CHAIN_SHA256, "used":True}
                       if CHAIN_OK else {"used":False, "reason":CHAIN_NOTE}),
  },
  "reference_base_verified": True,
  "acceptance_rule": ("Ensembl and dbSNP concordant on chr+pos; rsID not merged; "
                      "no multi-mapping; reference base matches; dbSNP previous-assembly "
                      "position matches Table 1 hg19 input"),
  "n_variants": int(len(m)),
  "n_accepted": int(m["accept"].sum()),
  "n_flagged":  int((~m["accept"]).sum()),
  "flagged": m.loc[~m["accept"],"rsid"].tolist(),
  "package_versions": {"python":sys.version.split()[0], "pandas":pd.__version__,
                       "requests":requests.__version__,
                       "pyliftover":_pyliftover_version()},
}
(OUTDIR/"Table1_liftover_provenance.json").write_text(json.dumps(provenance, indent=2))

print(f"{provenance['n_accepted']}/{provenance['n_variants']} variants accepted"
      + (f" - FLAGGED: {', '.join(provenance['flagged'])}" if provenance["flagged"] else "")
      + "\nwritten: Table1_GRCh38_coordinates.csv/.xlsx, Table1_liftover_full_audit.csv, "
        "Table1_liftover_provenance.json")

disp


## 9. Before updating the manuscript

1. Every row must read `pass`.
2. Check `rs11183620` falls inside the Fig. 5A window, chr12:46,815,609–46,835,708.
3. Note `rs6028984`: offset ≈ +1.37 Mb, far larger than the others. Probably real for
   chr20, but confirm the gene assignment (`LINC01370`) still holds at the new position —
   the nearest-gene annotation was made on GRCh37.
4. If `input_hg19_confirmed` is `False` anywhere, Table 1's **existing** hg19 coordinate
   was wrong, which is a separate correction.
5. Copy the Ensembl release and, if used, the chain SHA-256 from the provenance JSON into
   Supplementary Methods 4.8.
